# Signac 100M — Kaggle TPU qualification

This notebook runs bounded runtime canaries for the Signac M102 model, a same-depth FFN-aligned challenger, and a fully tiled challenger. It does **not** train on the research corpus. Random-token input checks hardware mechanics only. Keep outputs with the pinned source commit and Kaggle runtime version.

A passing device or model canary does not certify the production 8-replica launcher, qualified corpus, Citadel evaluation, or authorization for a capability experiment. See `docs/signac_100m/KAGGLE_TPU_RUNBOOK.md`.


In [ ]:
from pathlib import Path
import subprocess, sys

candidates = [Path('/kaggle/working/An-Ra-the-new-AGI-1'), *Path('/kaggle/input').glob('*/An-Ra-the-new-AGI-1'), *[p.parent for p in Path('/kaggle/input').glob('*/signac_100m')]]
roots = [p.resolve() for p in candidates if (p / 'signac_100m' / 'spec.py').is_file()]
if len(roots) != 1:
    raise RuntimeError(f'Expected exactly one attached repository snapshot; found {roots}')
REPO = roots[0]
sys.path.insert(0, str(REPO))
commit = subprocess.run(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True, capture_output=True)
COMMIT = commit.stdout.strip() if commit.returncode == 0 else 'UNAVAILABLE: record the full commit in notebook metadata'
print('Repository:', REPO)
print('Commit:', COMMIT)
assert COMMIT != 'UNAVAILABLE: record the full commit in notebook metadata', 'Use a pinned Git snapshot with .git metadata or fill the commit manually.'

In [ ]:
import importlib.metadata as md
import json, platform
import torch
import torch_xla
import torch_xla.runtime as xr
import torch_xla.core.xla_model as xm

device = xm.xla_device()
runtime = {
    'python': platform.python_version(), 'torch': torch.__version__,
    'torch_xla': md.version('torch_xla'), 'pjrt_device_type': xr.device_type(),
    'world_size': xr.world_size(), 'global_ordinal': xr.global_ordinal(),
    'global_device_count': xr.global_runtime_device_count(),
    'addressable_device_count': xr.addressable_runtime_device_count(),
    'device': str(device), 'platform': platform.platform(),
}
print(json.dumps(runtime, indent=2))
if str(runtime['pjrt_device_type']).upper() != 'TPU':
    raise RuntimeError('This notebook is for Kaggle TPU; no TPU result may be claimed for this runtime.')
if runtime['global_device_count'] < 1 or runtime['addressable_device_count'] < 1:
    raise RuntimeError('XLA runtime exposed no usable TPU devices.')

In [ ]:
from v5_training.target_preflight import PreflightConfig, run_preflight

# This PJRT process world can be size 1 while it sees an 8-device TPU. Check
# both values; this smoke preflight remains a single-process plumbing check.
target_receipt = run_preflight(PreflightConfig(
    expected_world_size=runtime['world_size'],
    expected_global_device_count=8,
))
print(json.dumps(target_receipt, indent=2))
if target_receipt.get('status') != 'PASS':
    raise RuntimeError(f'TPU runtime preflight failed: {target_receipt.get("reason", target_receipt.get("checks"))}')
Path('/kaggle/working/signac_100m_target_preflight.json').write_text(json.dumps(target_receipt, indent=2, sort_keys=True) + '\n')


In [ ]:
from signac_100m.spec import (
    MODEL_SPEC, TPU_DEPTH_PRESERVING_CHALLENGER, TPU_TILED_CHALLENGER, candidate_receipts,
)
from v5_training.target_canary import run_target_canary

receipts = candidate_receipts()
print(json.dumps({name: {'parameters': row['resources']['parameters_exact'], 'model_spec_sha256': row['model_spec_sha256'], 'attention_score_tensor_bytes_bf16_per_replica_per_active_layer': row['resources']['attention_score_tensor_bytes_bf16_per_replica_per_active_layer']} for name, row in receipts.items()}, indent=2))
assert MODEL_SPEC.parameter_receipt().total == 101_790_080
assert TPU_DEPTH_PRESERVING_CHALLENGER.parameter_receipt().total == 99_332_480
assert TPU_TILED_CHALLENGER.parameter_receipt().total == 100_303_104
from tools.signac_100m_preflight import build_report
static_report = build_report(target='tpu')
Path('/kaggle/working/signac_100m_static_preflight.json').write_text(json.dumps(static_report, indent=2, sort_keys=True) + '\n')
print('Static launch-gate report:', json.dumps({'verdict': static_report['verdict'], 'blockers': static_report['blockers'], 'training_authorized': static_report['training_authorized']}, indent=2))
assert static_report['training_authorized'] is False
out = Path('/kaggle/working/signac_100m_canaries')
small = run_target_canary(
    model_spec=MODEL_SPEC, device=device, workdir=out / 'm102_resume', seed=73011,
    batch_size=1, sequence_length=128, verify_checkpoint=True, verify_continuation=True,
)
print(json.dumps(small, indent=2))
if small['status'] != 'PASS':
    raise RuntimeError('Signac M102 checkpoint/continuation canary failed.')


## Full-context compute and memory canary

The next cell makes one 4,096-token forward/backward/update for each geometry with activation checkpointing. It does not create a second model copy for restore. It records cold first-step latency, which includes graph compilation; this is not steady-state throughput. The source reports an analytic BF16 attention-score tensor size per active layer but does not report reliable measured peak memory. A failure or OOM is useful qualification evidence. Inputs are synthetic tokens and cannot promote a capability claim.


In [ ]:
import gc

long_context = {}
for name, spec in [
    ('m102_primary', MODEL_SPEC),
    ('tpu_depth_preserving_challenger', TPU_DEPTH_PRESERVING_CHALLENGER),
    ('tpu_tiled_challenger', TPU_TILED_CHALLENGER),
]:
    gc.collect()
    xm.mark_step()
    try:
        receipt = run_target_canary(
            model_spec=spec, device=device, workdir=out / name / 'context_4096',
            seed=73012, batch_size=1, sequence_length=4096,
            verify_checkpoint=False, verify_continuation=False,
        )
    except Exception as exc:
        long_context[name] = {
            'status': 'FAIL', 'candidate': name,
            'model_spec_sha256': spec.sha256(),
            'parameter_count': spec.parameter_receipt().total,
            'error_type': type(exc).__name__, 'error': str(exc),
        }
        Path('/kaggle/working/signac_100m_long_context_receipts.json').write_text(
            json.dumps(long_context, indent=2, sort_keys=True) + '\n'
        )
        raise
    long_context[name] = receipt
    Path('/kaggle/working/signac_100m_long_context_receipts.json').write_text(
        json.dumps(long_context, indent=2, sort_keys=True) + '\n'
    )
    print(name, json.dumps(receipt, indent=2))
    if receipt['status'] != 'PASS':
        raise RuntimeError(f'{name} failed the 4096-context update canary')
    del receipt
    gc.collect()
    xm.mark_step()


## Preserve outputs and stop

Save `/kaggle/working/signac_100m_canaries/`, the preflight and long-context JSON receipts, the notebook, and runtime metadata as Kaggle Output. The 128-token canary checks model/Adam serialization and identical deterministic continuation; it does not test data cursor or per-rank XLA RNG resume. The 4096-token canaries exercise single-process updates and record cold compile-inclusive latency; they do not measure reliable peak memory or steady-state throughput. This does not qualify the production 8-replica launcher. Data, Citadel evaluation, multi-seed capability formation, and full production resume gates remain separate. Do not start the research run from these plumbing results alone.
